# E6 — Resolution ablation: source C at native 416 vs 640 upscale (YOLO26s, 15 epochs per arm)

**Question:** does the 640×640 normalisation step (letterbox + Lanczos upscale of source C's 11,110 416 px images)
destroy information in a way that affects results — particularly for small-object classes (goggles ~0.34% of image area, gloves, mask)?

**Design (two arms, isolating exactly one variable):**
- Uses only the **source-C image subset** (filtered via `manifest.csv`, keeping the group-aware split — no re-splitting, no leakage).
- **Arm `res640`**: the 640-upscaled C images as shipped in PPE-Unified v1, train/val/test at imgsz=640.
- **Arm `res416`**: the NATIVE 416 C images (Add Input the backup dataset `ppe-unified-v1-416c`); train/val/test at imgsz=416.
  Without the backup the notebook falls back to downscaling the 640 version to 416 (Lanczos) — *state the caveat in the report: these are not the native images.*
- Same YOLO26s, seed 42, lr0 0.01, optimizer auto, batch 16, 15 epochs, RFS off (to avoid confounding with E5).

**Output:** `results/e6_resolution.csv` — overall mAP@50/mAP@50-95 plus AP@50 for the small-object classes (goggles, no_goggles, gloves, no_gloves, mask)
per arm, plus the difference. Interpretation: if 640-upscale ≥ native-416, the normalisation is harmless (the model gets more pixel context);
if native-416 wins on the small classes, that is a real cost of normalisation and belongs in the Limitations section.

**Add Input:** `ppe-unified-v1` (+ `ppe-unified-v1-416c` if available). GPU T4 x2 · Internet ON. Estimated ~2.5 h. Checkpoint-resume supported.

In [ ]:
KEY_PREFIX = "e6_"
# ==================== CONFIG ====================
QUICK_TEST = False             # True = 1 epoch / a few batches as a pipeline smoke test; quick numbers must NOT go into the report
SEED = 42

import os, random, shutil, glob, json, time
WORK = "/kaggle/working"; os.makedirs(f"{WORK}/results", exist_ok=True)
STATE_PATH = f"{WORK}/state.json"
# RESUME: load state.json from THIS notebook's previous output (if Added as Input)
STATE = {}
for _st in glob.glob("/kaggle/input/**/state.json", recursive=True):
    try: _s = json.load(open(_st))
    except Exception: continue
    if any(k.startswith(KEY_PREFIX) for k in _s): STATE.update(_s); print("Resumed state:", list(_s))
def save_state():
    json.dump(STATE, open(STATE_PATH, "w"), ensure_ascii=False, indent=1)
def done(key): return key in STATE
random.seed(SEED)


In [ ]:
%pip install -q ultralytics
import torch; assert torch.cuda.is_available(), "Enable GPU T4 x2!"
print("GPU:", torch.cuda.get_device_name(0), "- OK")


In [ ]:
from pathlib import Path
import yaml, pandas as pd

def find_dataset_root():
    """Locate PPE_Unified_v1 under /kaggle/input (recursive glob — no hard-coded path)."""
    for c in glob.glob("/kaggle/input/**/data.yaml", recursive=True):
        root = os.path.dirname(c)
        if all(os.path.isdir(f"{root}/{s}/images") for s in ("train", "val", "test")):
            return root
    raise AssertionError("Dataset ppe-unified-v1 has not been Added as Input!")

def find_manifest():
    """manifest.csv traces each image back to its origin (source A/B/C) — produced by build_unified_dataset.py."""
    for c in glob.glob("/kaggle/input/**/manifest.csv", recursive=True):
        return pd.read_csv(c)
    raise AssertionError("manifest.csv not found in the input dataset!")

DS_ROOT = find_dataset_root()
MANIFEST = find_manifest()
print("Dataset:", DS_ROOT, "| manifest:", len(MANIFEST), "rows")
print(MANIFEST["source"].value_counts() if "source" in MANIFEST.columns else MANIFEST.columns.tolist())


In [ ]:
# ---------- Build the two arm datasets from source-C images only (split preserved) ----------
from PIL import Image

def build_subset_C(dst_name, resolution):
    """Copy the source-C images following the original split; resolution 640 => copy verbatim from PPE-Unified;
    resolution 416 => prefer the native-416 backup, otherwise fall back to Lanczos downscaling."""
    dst = f"{WORK}/{dst_name}"
    if os.path.isdir(dst): shutil.rmtree(dst)
    # look for the 416 backup (if the user Added it as Input)
    ROOT416 = None
    for c in glob.glob("/kaggle/input/**/416*/**/images", recursive=True):
        ROOT416 = os.path.dirname(c); break
    m = MANIFEST[MANIFEST["source"].str.upper().str.startswith("C")]
    n_fallback = 0
    for _, row in m.iterrows():
        split, fname = row["split"], row["file"]
        for sub in ("images", "labels"):
            os.makedirs(f"{dst}/{split}/{sub}", exist_ok=True)
        lbl = fname.rsplit(".", 1)[0] + ".txt"
        shutil.copy(f"{DS_ROOT}/{split}/labels/{lbl}", f"{dst}/{split}/labels/{lbl}")
        src_img = f"{DS_ROOT}/{split}/images/{fname}"
        if resolution == 640:
            shutil.copy(src_img, f"{dst}/{split}/images/{fname}")
        else:  # 416
            orig = glob.glob(f"{ROOT416}/**/{fname}", recursive=True) if ROOT416 else []
            if orig:
                shutil.copy(orig[0], f"{dst}/{split}/images/{fname}")
            else:  # FALLBACK: downscale — an approximation, not the native image
                Image.open(src_img).resize((416, 416), Image.LANCZOS).save(f"{dst}/{split}/images/{fname}")
                n_fallback += 1
    names = yaml.safe_load(open(f"{DS_ROOT}/data.yaml"))["names"]
    dyaml = f"{dst}/data.yaml"
    yaml.dump({"path": dst, "train": "train/images", "val": "val/images", "test": "test/images",
               "nc": len(names), "names": names}, open(dyaml, "w"))
    if n_fallback: print(f"CAVEAT: {n_fallback} images used the downscale fallback (no native-416 backup available)")
    return dyaml


In [ ]:
# ---------- Train + eval 2 arm ----------
from ultralytics import YOLO
import pandas as pd
SMALL = ["goggles", "no_goggles", "gloves", "no_gloves", "mask"]
EPOCHS = 1 if QUICK_TEST else 15
rows = []
for arm, res in [("res640", 640), ("res416", 416)]:
    key = f"e6_{arm}"
    if done(key): rows.append(STATE[key]); print("skip:", key); continue
    dyaml = build_subset_C(f"dsC_{arm}", res)
    t0 = time.time()
    m = YOLO("yolo26s.pt")
    m.train(data=dyaml, epochs=EPOCHS, imgsz=res, batch=16, seed=SEED, lr0=0.01,
            project=f"{WORK}/runs", name=key, exist_ok=True, plots=False, verbose=False)
    t = m.val(data=dyaml, split="test", imgsz=res, batch=16)
    names = list(t.names.values())
    per = {f"AP50_{n}": round(float(a), 4) for n, a in zip(names, t.box.ap50)}
    rec = {"arm": arm, "imgsz": res, "mAP50_test": round(float(t.box.map50), 4),
           "mAP50_95_test": round(float(t.box.map), 4),
           **{k: per.get(k) for k in [f"AP50_{c}" for c in SMALL]},
           "min": round((time.time() - t0) / 60, 1)}
    STATE[key] = rec; save_state(); rows.append(rec)
df = pd.DataFrame(rows); df.to_csv(f"{WORK}/results/e6_resolution.csv", index=False)
print(df.to_string(index=False))


In [ ]:
# ==================== CLEAN UP OUTPUT: no dataset bundled ====================
KEEP = {"results", "state.json", "runs"}
for item in os.listdir(WORK):
    if item in KEEP: continue
    p = os.path.join(WORK, item)
    shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else os.remove(p)
print("Final output:", sorted(os.listdir(WORK)))
